# RSNA Knee Abnormality Detection — Phase 4C gold-only second-reader pilot

Internet-off Kaggle notebook. **Do not enable internet. GPU is required.**
This notebook scores the **58 gold studies only**. It does **not** process
the 4,349 report-only studies, does **not** train an image model, does **not**
create image folds, and does **not** submit to the leaderboard.

Attach checksummed `microsoft/Phi-3.5-mini-instruct` weights as a Kaggle Model
or private dataset. Do not download weights. Do not print report text.

Software-fixture executions must keep `scientific_evaluation=false`.


In [ ]:
# Bootstrap: stdlib only until the package is importable.
# Copies/extracts source under /kaggle/working. Never writes under /kaggle/input.
from __future__ import annotations

import os
import shutil
import sys
import zipfile
from pathlib import Path

KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")
COMPETITION_SLUG = "rsna-knee-abnormality-detection"
SRC_DST = KAGGLE_WORKING / "rsna_knee_src"
OUTPUT_DIR = KAGGLE_WORKING / "rsna_second_reader_gold_pilot"
MARKERS = ("knee/config.py", "knee/second_reader_pilot.py")

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"


def _looks_like_source(root: Path) -> bool:
    return all((root / marker).is_file() for marker in MARKERS)


def _safe_extract(zip_path: Path, dest: Path) -> None:
    dest.mkdir(parents=True, exist_ok=True)
    dest_resolved = dest.resolve()
    with zipfile.ZipFile(zip_path) as zf:
        for info in zf.infolist():
            name = info.filename
            if not name or name.endswith("/"):
                continue
            target = (dest / name).resolve()
            try:
                target.relative_to(dest_resolved)
            except ValueError as exc:
                raise RuntimeError(f"refusing zip path that escapes {dest}: {name}") from exc
            target.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(info) as src, target.open("wb") as out:
                shutil.copyfileobj(src, out)


def _copy_allowlisted(src_root: Path, dest: Path) -> None:
    dest.mkdir(parents=True, exist_ok=True)
    for dirname in ("knee", "scripts", "configs", "notebooks"):
        src = src_root / dirname
        if src.is_dir():
            shutil.copytree(src, dest / dirname, dirs_exist_ok=True)
    for filename in ("pyproject.toml", "requirements.txt", "README.md"):
        src = src_root / filename
        if src.is_file():
            shutil.copy2(src, dest / filename)


def _zip_has_source(zip_path: Path) -> bool:
    try:
        with zipfile.ZipFile(zip_path) as zf:
            names = zf.namelist()
    except (OSError, zipfile.BadZipFile):
        return False
    return any(any(name.endswith(marker) for marker in MARKERS) for name in names)


SKIP_WALK_DIRS = {"train_series", "test_series", "__pycache__", ".git", "competitions"}


def _walk_input_for_source(root: Path, max_depth: int = 6):
    zips: list[Path] = []
    dirs: list[Path] = []
    try:
        root_res = root.resolve()
    except OSError:
        return zips, dirs
    for dirpath, dirnames, filenames in os.walk(root):
        current = Path(dirpath)
        try:
            rel = current.resolve().relative_to(root_res)
            depth = 0 if str(rel) == "." else len(rel.parts)
        except (OSError, ValueError):
            dirnames[:] = []
            continue
        dirnames[:] = sorted(
            d for d in dirnames if d not in SKIP_WALK_DIRS and d != COMPETITION_SLUG
        )
        if depth > max_depth:
            dirnames[:] = []
            continue
        if _looks_like_source(current):
            dirs.append(current)
        for name in filenames:
            if name.lower().endswith(".zip"):
                zip_path = current / name
                if _zip_has_source(zip_path):
                    zips.append(zip_path)
    return zips, dirs


def locate_and_extract_source() -> Path | None:
    if _looks_like_source(SRC_DST):
        return SRC_DST
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if _looks_like_source(candidate):
            if KAGGLE_WORKING.exists() and candidate != SRC_DST:
                _copy_allowlisted(candidate, SRC_DST)
                if _looks_like_source(SRC_DST):
                    return SRC_DST
            return candidate
    if not KAGGLE_INPUT.is_dir():
        return None
    zips: list[Path] = []
    dirs: list[Path] = []
    for child in sorted(KAGGLE_INPUT.iterdir()):
        if child.name == COMPETITION_SLUG or child.name == "competitions":
            continue
        if child.is_file() and child.suffix.lower() == ".zip" and _zip_has_source(child):
            zips.append(child)
        elif child.is_dir():
            found_zips, found_dirs = _walk_input_for_source(child)
            zips.extend(found_zips)
            dirs.extend(found_dirs)
    if zips:
        print(f"extracting_source_zip={zips[0]}")
        _safe_extract(zips[0], SRC_DST)
        if _looks_like_source(SRC_DST):
            return SRC_DST
        for nested in sorted(p for p in SRC_DST.iterdir() if p.is_dir()):
            if _looks_like_source(nested):
                return nested
    if dirs:
        print(f"copying_source_tree={dirs[0]}")
        _copy_allowlisted(dirs[0], SRC_DST)
        if _looks_like_source(SRC_DST):
            return SRC_DST
        return dirs[0]
    return None


SOURCE_ROOT = locate_and_extract_source()
if SOURCE_ROOT is not None:
    resolved = str(SOURCE_ROOT.resolve())
    if resolved not in sys.path:
        sys.path.insert(0, resolved)
    print(f"source_root={SOURCE_ROOT}")
else:
    print("source_root=not_found (will rely on an already importable knee package)")

try:
    from knee.kaggle_runtime import detect_kaggle_paths, KaggleLayout
    from knee.second_reader_pilot import KAGGLE_PILOT_OUTPUT_DIRNAME, run_second_reader_gold_pilot
except ImportError as exc:
    raise SystemExit(
        "PILOT STOPPED: could not import knee. Attach the private source dataset. "
        f"Original error: {exc}"
    ) from exc

print("internet_used=False gpu_required=True training_run=False process_report_only=False")
print(detect_kaggle_paths(KaggleLayout(output_dirname=KAGGLE_PILOT_OUTPUT_DIRNAME)))


In [ ]:
# Gold-only second-reader pilot. Do not print Report text or identifiers.
from pathlib import Path

from knee.kaggle_runtime import KaggleLayout, preflight
from knee.report_labels import assert_hard_weak_label_training_forbidden
from knee.second_reader_pilot import run_second_reader_gold_pilot

assert_hard_weak_label_training_forbidden()

PILOT_OUT = "rsna_second_reader_gold_pilot"
layout = KaggleLayout(output_dirname=PILOT_OUT)
DATA_ROOT = Path(os.environ["RSNA_DATA_ROOT"]) if os.environ.get("RSNA_DATA_ROOT") else None
OUT_DIR = Path(os.environ["RSNA_SECOND_READER_OUTPUT_DIR"]) if os.environ.get("RSNA_SECOND_READER_OUTPUT_DIR") else layout.output_dir
WEIGHTS = os.environ.get("SECOND_READER_WEIGHTS")

pre = preflight(layout=layout, data_root=DATA_ROOT, output_dir=OUT_DIR)
print(pre.message)
print("data_source=", pre.data_source)
print("scientific_evaluation=", pre.scientific_evaluation)
print("Competition top-level (not a DICOM walk):", pre.top_level_competition)

cfg = None
if SOURCE_ROOT is not None:
    cfg = str(Path(SOURCE_ROOT) / "configs" / "second_reader_pilot.yaml")

result = run_second_reader_gold_pilot(
    layout=layout,
    data_root=DATA_ROOT if DATA_ROOT is not None else pre.data_root,
    output_dir=OUT_DIR,
    data_source=None if DATA_ROOT is None else os.environ.get("RSNA_REPORT_LABEL_DATA_SOURCE"),
    config_path=cfg,
    weights_path=WEIGHTS,
)

print("status=", result["status"])
print("DATA_SOURCE=", result["data_source"])
print("scientific_evaluation=", result["scientific_evaluation"])
print("claim_scientific_performance=", result.get("claim_scientific_performance"))
print("training_run=", result["training_run"], "folds_created=", result["folds_created"])
print("n_gold_studies=", result.get("n_gold_studies"))
print("n_report_only_studies_processed=", result.get("n_report_only_studies_processed"))
print("download_weights=", result.get("download_weights"))
print("internet_used=", result.get("internet_used"))
    print("inference_backend=", result.get("inference_backend"))
print("malformed_output_rate=", result.get("malformed_output_rate"))
assert result.get("n_report_only_studies_processed") in {0, None}
assert result.get("training_run") is False
assert result.get("folds_created") is False
assert result.get("download_weights") is False
assert result.get("internet_used") is False
if result.get("data_source") == "real_kaggle_mount":
    assert result.get("inference_backend") == "local_transformers"
    assert result.get("n_gold_studies") == 58
    assert result.get("scientific_evaluation") is True
    assert result.get("n_report_only_studies_processed") == 0


In [ ]:
# Display sanitized aggregates only. Never read or print private row-level files.
from pathlib import Path
import json

out = Path(result["output_dir"])
sanitized = out / "sanitized"
print("sanitized_dir=", sanitized)
print("DATA_SOURCE=", result["data_source"])
print("scientific_evaluation=", result["scientific_evaluation"])

summary_path = sanitized / "second_reader_gold_pilot_summary.md"
print(summary_path.read_text(encoding="utf-8")[:12000])

for name in (
    "freeze_manifest.json",
    "second_reader_gold_metrics.json",
    "runtime_metrics.json",
    "pilot_acceptance.json",
    "private_output_hashes.json",
):
    path = sanitized / name
    print(f"\n=== {name} ===")
    print(path.read_text(encoding="utf-8")[:4000])

print("private hashes recorded:", sorted((result.get("private_hashes") or {}).keys()))
print("n_report_only_studies_processed=", result.get("n_report_only_studies_processed"))
